# JS08 — Lab 3: Eksperimen Hyperparameter HDBSCAN

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** melihat pengaruh `min_cluster_size`, `min_samples`, dan `cut_distance` terhadap jumlah cluster, jumlah noise, dan kualitas pemisahan (*Silhouette Score*, *Davies-Bouldin Index*).

**Prasyarat:** tidak perlu dataset eksternal — data *multiscale* (4 cluster) dibangkitkan ulang di cell persiapan, jadi notebook ini bisa dijalankan mandiri tanpa Lab 2.


# Lab 3 — Eksperimen Hyperparameter HDBSCAN

Fokus pada pengaruh `min_cluster_size`, `min_samples`, dan `cut_distance` terhadap jumlah cluster, penanganan noise, dan kualitas pemisahan.

> Dataset yang dipakai sama dengan Lab 2 Langkah 5 (*multiscale*, 4 cluster). Datanya dibuat ulang di cell pertama agar Lab 3 bisa dijalankan mandiri tanpa harus menjalankan Lab 2 terlebih dahulu.

## Persiapan — Data & Fungsi `plot()`

In [ ]:
# Siapkan ulang data multiscale + fungsi plot (mandiri, tidak bergantung Lab 2)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hdbscan
from sklearn.datasets import make_blobs

centers = [[-0.85, -0.85], [-0.85, 0.85], [3, 3], [3, -3]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.2, 0.35, 1.35, 1.35], random_state=0
)

def plot(X, labels, probabilities=None, parameters=None, ground_truth=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    labels = labels if labels is not None else np.ones(X.shape[0])
    probabilities = probabilities if probabilities is not None else np.ones(X.shape[0])
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    proba_map = {idx: probabilities[idx] for idx in range(len(labels))}

    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = [0, 0, 0, 1]
        class_index = (labels == k).nonzero()[0]
        for ci in class_index:
            ax.plot(
                X[ci, 0], X[ci, 1],
                "x" if k == -1 else "o",
                markerfacecolor=tuple(col),
                markeredgecolor="k",
                markersize=4 if k == -1 else 1 + 5 * proba_map[ci],
            )

    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    preamble = "True" if ground_truth else "Estimated"
    title = f"{preamble} number of clusters: {n_clusters_}"
    if parameters is not None:
        parameters_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
        title += f" | {parameters_str}"
    ax.set_title(title)
    plt.tight_layout()

plot(X, labels=labels_true, ground_truth=True)

### [tambahan] Fungsi evaluasi

Satu fungsi bantu untuk menghitung jumlah cluster, jumlah noise, *Silhouette Score*, dan *Davies-Bouldin Index*. Titik noise (label `-1`) dikeluarkan dari perhitungan metrik, karena metrik internal mengasumsikan setiap titik anggota sebuah cluster.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

def evaluasi(X, labels, nama_param, **param):
    labels = np.asarray(labels)
    mask = labels != -1
    n_cluster = len(set(labels[mask]))
    baris = {
        **param,
        'n_cluster': n_cluster,
        'n_noise': int((~mask).sum()),
        'silhouette': np.nan,
        'dbi': np.nan,
    }
    # metrik hanya valid bila ada >= 2 cluster dan setiap cluster berisi > 1 titik
    if n_cluster >= 2 and np.bincount(labels[mask]).min() > 1:
        baris['silhouette'] = silhouette_score(X[mask], labels[mask])
        baris['dbi'] = davies_bouldin_score(X[mask], labels[mask])
    return baris

## Langkah 1 — Eksperimen Hyperparameter `min_cluster_size`

In [ ]:
PARAM = ({"min_cluster_size": 5}, {"min_cluster_size": 3}, {"min_cluster_size": 25})
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = hdbscan.HDBSCAN(**param).fit(X)
    plot(X, hdb.labels_, hdb.probabilities_, param, ax=axes[i])
plt.show()

In [ ]:
# [tambahan] Rekap metrik tiap nilai min_cluster_size
rows = []
for param in PARAM:
    h = hdbscan.HDBSCAN(**param).fit(X)
    rows.append(evaluasi(X, h.labels_, 'min_cluster_size', **param))
pd.DataFrame(rows).round(4)

## Langkah 2 — Eksperimen Hyperparameter `min_samples`

`min_cluster_size=20` dibuat tetap, nilai `min_samples` divariasikan.

In [ ]:
PARAM = (
    {"min_cluster_size": 20, "min_samples": 5},
    {"min_cluster_size": 20, "min_samples": 3},
    {"min_cluster_size": 20, "min_samples": 25},
)
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = hdbscan.HDBSCAN(**param).fit(X)
    plot(X, hdb.labels_, hdb.probabilities_, param, ax=axes[i])
plt.show()

In [ ]:
# [tambahan] Rekap metrik tiap nilai min_samples
rows = []
for param in PARAM:
    h = hdbscan.HDBSCAN(**param).fit(X)
    rows.append(evaluasi(X, h.labels_, 'min_samples', **param))
pd.DataFrame(rows).round(4)

## Langkah 3 — DBSCAN Clustering dari Pohon HDBSCAN (`cut_distance`)

`dbscan_clustering()` memotong pohon hirarki HDBSCAN pada jarak tertentu, sehingga menghasilkan pelabelan ala DBSCAN dari model yang sudah dilatih.

In [ ]:
PARAM = (
    {"cut_distance": 0.1},
    {"cut_distance": 0.5},
    {"cut_distance": 1.0},
)
hdb = hdbscan.HDBSCAN().fit(X)
fig, axes = plt.subplots(len(PARAM), 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    labels = hdb.dbscan_clustering(**param)
    plot(X, labels, hdb.probabilities_, param, ax=axes[i])
plt.show()

In [ ]:
# [tambahan] Rekap metrik tiap nilai cut_distance
rows = []
for param in PARAM:
    lab = hdb.dbscan_clustering(**param)
    rows.append(evaluasi(X, lab, 'cut_distance', **param))
pd.DataFrame(rows).round(4)

## Langkah 4 — Evaluasi dengan **Silhouette Score**

- Mendekati **1** → cluster terpisah dengan baik.
- Mendekati **0** → cluster saling bertumpang tindih.
- **Negatif** → banyak titik kemungkinan salah kelompok.

In [ ]:
from sklearn.metrics import silhouette_score

# Menghitung Silhouette Score untuk hasil clustering HDBSCAN
sil_score = silhouette_score(X, hdb.labels_)
print(f"Silhouette Score: {sil_score}")

## Langkah 5 — Evaluasi dengan **Davies-Bouldin Index**

- Nilai **lebih rendah** → cluster lebih terpisah dan homogen (lebih baik).
- Nilai **lebih tinggi** → cluster saling tumpang tindih.

In [ ]:
from sklearn.metrics import davies_bouldin_score

# Menghitung Davies-Bouldin Index untuk hasil clustering HDBSCAN
dbi_score = davies_bouldin_score(X, hdb.labels_)
print(f"Davies-Bouldin Index: {dbi_score}")

## Langkah 6 — Visualisasi Hasil Evaluasi

> **Catatan:** pada modul, `hdb.labels_` di cell evaluasi masih merujuk objek `hdb` yang terakhir dilatih (default `hdbscan.HDBSCAN()`, hasilnya sama dengan Langkah 4 Lab 2). Di sini `hdb` adalah model default yang sama dari Langkah 3 (`cut_distance` tidak mengubah `labels_`), sehingga hasil evaluasinya konsisten.

In [ ]:
# Misalnya kita ingin membandingkan DBI dan Silhouette Score untuk beberapa eksperimen
scores = {
    "Silhouette Score": sil_score,
    "Davies-Bouldin Index": dbi_score
}

# Plot hasil evaluasi
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(scores.keys(), scores.values(), color=['skyblue', 'salmon'])
ax.set_title("Evaluasi Hasil Clustering")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

---
## Kesimpulan Lab 3

**Pengaruh `min_cluster_size`:** TBD

**Pengaruh `min_samples`:** TBD

**Pengaruh `cut_distance`:** TBD

**Kombinasi parameter terbaik menurut Silhouette/DBI:** TBD

*(Gunakan tabel rekap `[tambahan]` sebagai bukti angka.)*